# 01 — Raw Data Exploration

**Project:** World Forest Change & GDP Correlation Analysis
**Database:** db_forestgdp (MSSQL Server)
**Team:** [Kucerova Kristina], [Kmetova Barbara]
**Date:** May 2026

## Purpose
This notebook explores the raw tables loaded from Kaggle and other sources before any
transformations are applied. The goal is to understand the structure,
spot data quality issues and document findings that will drive
decisions in the next step (02_dim_tables).

## Raw tables
| raw.Forest_year | FAO / Our World in Data | Forest coverage % per country per year |
| raw.Forest_watch| Forest Watch | Kaggle | Deforestation and forest disturbance data |
| raw.GDP | World Bank / Kaggle | GDP values per country per year |
| raw.stg_deforest_aforest | Kaggle | Deforestation and afforestation staging data |
| raw.stg_gdp_income | World Bank | Kaggle | Country income group classifications |

## What we are looking for
- Row counts and year ranges per table
- Missing or empty country codes
- Duplicates
- Null values in key columns
- Country code consistency across tables 
- Check which countries in Forest_year have no matching region in the mapping table.

## Findings
- 5 raw tables ranging from 217 to 17,290 rows
- Analysis range set to 1990—2024 where Forest_year and GDP overlap
- Forest_year has entities without ISO3 codes (EU, England, Scotland etc.) — will be handled with AGG_ prefix in dim_country. Also World with OWID_WRL code — will be flagged in dim_country
- GDP tables contains one indicator only: GDP per capita (current US$)
- GDP NULLs expected for pre-1990 years — not an issue for our analysis
- 4 countries have forest data but no GDP match: French Guiana, Taiwan, Western Sahara, World as World Bank doesn't recongnise them as countries — will be kept in dim_country but excluded from correlation analysis
- No duplicates, no nulls in key columns across Forest_year and GDP
- raw.forest_policy subregion column contains empty strings instead of NULL for 89 countries. Fixed in 02_dim_tables.
- Taiwan (TWN) and Western Sahara (ESH) have no region — not recognised by World Bank (will be fixed manually in 02_dim_tables)


In [21]:
-- Row counts
SELECT 'Forest_year' AS table_name,
    COUNT(*) AS row_count 
FROM raw.Forest_year

UNION ALL
SELECT 'Forest_watch' AS table_name, 
    COUNT(*) AS row_count 
FROM raw.Forest_watch

UNION ALL
SELECT 'GDP' AS table_name, 
    COUNT(*) AS row_count
FROM raw.GDP

UNION ALL
SELECT 'stg_deforest_aforest' AS table_name, 
    COUNT(*) AS row_count 
FROM raw.stg_deforest_aforest

UNION ALL
SELECT 'stg_gdp_income' AS table_name,
    COUNT(*) AS row_count 
FROM raw.stg_gdp_income;

(5 rows affected)

table_name           | row_count
---------------------+----------
Forest_year          | 7970     
Forest_watch         | 5720     
GDP                  | 17290    
stg_deforest_aforest | 217      
stg_gdp_income       | 217      
(5 rows)

Total execution time: 00:00:00.995

In [ ]:
-- Year ranges across all tables
SELECT 'Forest_year' AS table_name, 
    MIN(Year) AS year_from, 
    MAX(Year) AS year_to 
FROM raw.Forest_year

UNION ALL

SELECT 'GDP',
    MIN(Year),
    MAX(Year)
FROM raw.GDP;

(2 rows affected)

table_name  | year_from | year_to
------------+-----------+--------
Forest_year | 1000      | 2025   
GDP         | 1960      | 2024   
(2 rows)

Total execution time: 00:00:00.051

In [ ]:
-- Sample rows from each table
SELECT TOP 5 * FROM raw.Forest_year;
SELECT TOP 5 * FROM raw.Forest_watch;
SELECT TOP 5 * FROM raw.GDP;
SELECT TOP 5 * FROM raw.stg_deforest_aforest;
SELECT TOP 5 * FROM raw.stg_gdp_income;

(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)

Country     | Year | Forest_percentage | Code | Notes
------------+------+-------------------+------+------
Afghanistan | 1990 | 1,8543152         | AFG  |      
Afghanistan | 1991 | 1,8543152         | AFG  |      
Afghanistan | 1992 | 1,8543152         | AFG  |      
Afghanistan | 1993 | 1,8543152         | AFG  |      
Afghanistan | 1994 | 1,8543152         | AFG  |      
(5 rows)

Country | Year | Forest_Area_km2 | Land_Area_km2 | Forest_Cover_Pct | Annual_Deforestation_Rate | Annual_Afforestation_Rate | Total_Carbon_Stock_Tonnes | Primary_Driver_of_Change
--------+------+-----------------+---------------+------------------+---------------------------+---------------------------+---------------------------+-------------------------
Brazil  | 2000 | 5468301         | 8515767       | 64,21            | 0                         | 0                         | 87197411954               | Initial S

In [ ]:
-- Check for missing or empty country codes in Forest_year
SELECT DISTINCT Country, Code
FROM raw.Forest_year
WHERE Code IS NULL OR Code = ''
ORDER BY Country;

(7 rows affected)

Country                       | Code
------------------------------+-----
England                       |     
European Union (27)           |     
High-income countries         |     
Low-income countries          |     
Lower-middle-income countries |     
Scotland                      |     
Upper-middle-income countries |     
(7 rows)

Total execution time: 00:00:01.003

In [ ]:
-- Check for codes longer than 3 characters (typically OWID aggregates)
SELECT DISTINCT Code, Country
FROM raw.Forest_year
WHERE LEN(Code) > 3
ORDER BY Code;

(1 row affected)

Code     | Country
---------+--------
OWID_WRL | World  
(1 row)

Total execution time: 00:00:06.504

In [ ]:
-- Check for duplicate rows in Forest_year
SELECT Code, Country, Year, COUNT(*) AS occurrences
FROM raw.Forest_year
GROUP BY Code, Country, Year
HAVING COUNT(*) > 1;


(0 rows affected)

(0 rows)

Total execution time: 00:00:00.614

In [ ]:
-- Check for nulls in key columns of Forest_year
SELECT
    COUNT(*) - COUNT(Code)              AS null_code,
    COUNT(*) - COUNT(Country)           AS null_country,
    COUNT(*) - COUNT(Year)              AS null_year,
    COUNT(*) - COUNT(Forest_percentage) AS null_forest_pct
FROM raw.Forest_year;

(1 row affected)

null_code | null_country | null_year | null_forest_pct
----------+--------------+-----------+----------------
0         | 0            | 0         | 0              
(1 row)

Total execution time: 00:00:00.987

In [ ]:
-- Check for nulls in key columns of GDP
SELECT
    COUNT(*) - COUNT([Country Name])    AS null_country,
    COUNT(*) - COUNT([Country Code])    AS null_country_code,
    COUNT(*) - COUNT(Year)              AS null_year,
    COUNT(*) - COUNT(GDP)               AS null_gdp
FROM raw.GDP;

(1 row affected)

null_country | null_country_code | null_year | null_gdp
-------------+-------------------+-----------+---------
0            | 0                 | 0         | 2729    
(1 row)

Total execution time: 00:00:00.942

In [ ]:
-- Check if country codes in Forest_year exist in GDP
SELECT DISTINCT f.Code, f.Country
FROM raw.Forest_year f
LEFT JOIN raw.GDP g ON g.[Country Code] = f.Code
WHERE g.[Country Code] IS NULL
AND f.Code IS NOT NULL
AND f.Code <> ''
ORDER BY f.Country;

(4 rows affected)

Code     | Country       
---------+---------------
GUF      | French Guiana 
TWN      | Taiwan        
ESH      | Western Sahara
OWID_WRL | World         
(4 rows)

Total execution time: 00:00:00.744

In [20]:
-- Get all distinct geographic groups of region and subregion from dataset
SELECT DISTINCT subregions, regions
FROM raw.Forest_Policy_Legislation;

(12 rows affected)

subregions                  | regions                  
----------------------------+--------------------------
                            | Europe                   
                            | Oceania                  
                            | South America            
Caribbean                   | North and Central America
Central America             | North and Central America
East Asia                   | Asia                     
Eastern and Southern Africa | Africa                   
North America               | North and Central America
Northern Africa             | Africa                   
South and Southeast Asia    | Asia                     
Western and Central Africa  | Africa                   
Western and Central Asia    | Asia                     
(12 rows)

Total execution time: 00:00:00.138